In [ ]:
conda install -c conda-forge numpy pandas matplotlib librosa plotly polars tqdm jupyter
pip install scikit-learn scipy torch torchaudio transformers datasets accelerate soundfile ipython modelscope

In [ ]:
import os
import librosa
import numpy as np
import random
import pandas as pd
from IPython.display import display, Audio
import string
from tqdm.notebook import tqdm
import collections
from pathlib import Path
import polars as pl
import plotly.graph_objs as go

import torch
import torchaudio
from torch.utils.data import Dataset, DataLoader

from numpy.fft import fft, fftfreq
from scipy.signal import butter, lfilter

from datasets import load_dataset, Audio, ClassLabel, DatasetDict
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    pipeline
)

from modelscope.pipelines import pipeline
from modelscope.utils.constant import Tasks
from transformers import AutoModelForCausalLM, AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Data

#### Load audio

In [ ]:
# Random data

folder = "/content/kaggle_competition/speechs/speechs/test"
wav_files = [f for f in os.listdir(folder) if f.endswith(".wav")]
random_file = random.choice(wav_files)
waveform, sample_rate = torchaudio.load(os.path.join(folder, random_file))
resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
waveform = resampler(waveform)

print(waveform)
print(sample_rate)

In [ ]:
waveform = waveform.squeeze()

fft_output = torch.fft.rfft(waveform)

magnitude_spectrum = torch.abs(fft_output)

num_bins = len(magnitude_spectrum)
frequencies = torch.linspace(0, sample_rate / 2, num_bins)

# Convert to numpy for Plotly (Plotly can handle tensors too, but numpy is common)
frequencies_np = frequencies.numpy()
magnitude_spectrum_np = magnitude_spectrum.numpy()

# 4. Plotting with Plotly
fig = go.Figure(data=go.Scatter(x=frequencies_np, y=magnitude_spectrum_np, mode='lines'))

fig.update_layout(
    title='FFT Magnitude Spectrum',
    xaxis_title='Frequency (Hz)',
    yaxis_title='Magnitude',
    hovermode='x unified', # Shows a tooltip across the x-axis for all traces
    template='plotly_white' # A clean white background template
)

fig.show()

In [ ]:
#Band pass filter function
def get_dominant_frequency(signal, sample_rate=16000):
    n = len(signal)
    yf = np.abs(fft(signal))[:n // 2]  # Take positive frequencies
    xf = fftfreq(n, 1 / sample_rate)[:n // 2]  # Frequency bins

    idx = np.argmax(yf)  # Index of max amplitude
    dominant_freq = xf[idx]
    return dominant_freq

def butter_bandpass(lowcut, highcut, fs, order=5):
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype='band')
    return b, a

def apply_bandpass_filter(data, lowcut, highcut, fs, order=5):
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    y = lfilter(b, a, data)
    return y

In [ ]:
def noise_reduce(waveform):
  #Change to the right format
  wavelet_clean = waveform.squeeze()

  #3.Band pass filter
  highest_freq = get_dominant_frequency(wavelet_clean, 16000)
  lowcut = max(300, highest_freq - 1000)
  wavelet_clean = apply_bandpass_filter(wavelet_clean, lowcut, max(4000, highest_freq + 1000), sample_rate)

  return wavelet_clean

# Model

In [ ]:
MODEL_NAME = "nectec/Pathumma-llm-audio-1.0.0"
lang = "th"

device = 0 if torch.cuda.is_available() else "cpu"

pipe = pipeline(
    task="automatic-speech-recognition",
    model=MODEL_NAME,
    chunk_length_s=20,
    device=device,
    batch_size=8,
)

In [ ]:
!rm -rf /project/ai901504-ai0004/501641_Big/week4/denoise
!mkdir /project/ai901504-ai0004/501641_Big/week4/denoise
!mkdir /project/ai901504-ai0004/501641_Big/week4/denoise/train
!mkdir /project/ai901504-ai0004/501641_Big/week4/denoise/test

### This code function are
1.select the set you want to transcript (train/test) \
2.denoise \
3.use asr to transcript \
4.create as a df of transcript from audio

In [ ]:
train_df = pl.read_csv('/project/ai901504-ai0004/501641_Big/week4/Human_Labor_train.csv')
test_df = pl.read_csv('//project/ai901504-ai0004/501641_Big/week4/test.csv')

predict_data = 'test'
predict_df = test_df
num_files = 300

In [ ]:
input_paths = Path(f'/project/ai901504-ai0004/501641_Big/week4/speechs/speechs/{predict_data}').glob('*.wav')
output_path = Path(f'/project/ai901504-ai0004/501641_Big/week4/denoise/{predict_data}')

In [ ]:
for path in tqdm(input_paths, total=num_files):
  waveform, sample_rate = torchaudio.load(str(path))
  waveform = noise_reduce(waveform)
  waveform = waveform.unsqueeze(0)
  torchaudio.save(output_path/path.name, waveform, sample_rate)

In [ ]:
output_path = Path(f'/project/ai901504-ai0004/501641_Big/week4/denoise/{predict_data}')

output_paths = list(output_path.glob('*.wav'))

file_names = [path.stem for path in output_paths]
out_paths = [str(path) for path in output_paths]

In [ ]:
result = pipe(out_paths)

In [ ]:
del pipe

In [ ]:
import gc

torch.cuda.empty_cache()
gc.collect()

In [ ]:
result[:10]

In [ ]:
result_dict = {}
i = 0

for name in file_names:
  result_dict[name] = result[i]['text']
  i += 1

predict_df = predict_df.with_columns(
    pl.col("id").replace(result_dict).alias("denoised_text")
)

In [ ]:
predict_df.head()